# Fine-tune an MMS-1B adapter for Nuer (CTC)
**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 1: GPU CHECK + DEPENDENCIES
import torch
assert torch.cuda.is_available(), "No GPU found. Runtime > Change runtime type > T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q transformers datasets accelerate soundfile librosa jiwer evaluate torchaudio safetensors

GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.8 MB/s eta 0:00:00


In [ ]:
# CELL 2: MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELL 3: DATASET EXTRACTION, PATH RESOLUTION AND TRANSCRIPT CLEANING
import os, re
import zipfile
import pandas as pd
import unicodedata
from collections import Counter
from sklearn.model_selection import train_test_split

# Pointing to the new zip file containing Acts 1 to 8 so running from top-to-bottom doesn't crash
zip_path = '/content/drive/MyDrive/corpus/Acts_1_to_8.zip'
extract_path = '/content/dataset_acts_1_8'

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# We will use the combined dataset extraction for execution safety
import glob
metadata_files = glob.glob(os.path.join(extract_path, '**/metadata.csv'), recursive=True)

df_list = []
for file_path in metadata_files:
    temp_df = pd.read_csv(file_path)
    parent_dir = os.path.dirname(file_path)
    temp_df['absolute_dir_path'] = parent_dir
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

# Characters that are ALWAYS kept, even though they are punctuation/symbols:
#   '/'  = negation marker at the start of a word (e.g. "/Ci̱" is NOT the same word as "Ci̱")
PROTECTED = {'/'}

removed_chars = Counter()

def clean_and_normalize(text):
    if not isinstance(text, str):
        return ""
    # NFC keeps precomposed diacritics consistent (must also be used at inference time)
    text = unicodedata.normalize('NFC', text)
    kept = []
    for ch in text:
        cat = unicodedata.category(ch)
        if (
            ch in PROTECTED          # the negation slash
            or ch.isalpha()          # letters incl. ɛ ɔ ŋ ɣ ä ë ö ...
            or cat.startswith('M')   # combining marks (e.g. U+0331 macron below, U+0308 diaeresis)
            or ch.isspace()
        ):
            kept.append(ch)
        else:
            removed_chars[ch] += 1   # punctuation, digits, symbols etc. get dropped (and reported below)
    text = ''.join(kept)
    return re.sub(r'\s+', ' ', text).strip()

df['transcription_clean'] = df['transcription'].apply(clean_and_normalize)

# ---- Sanity checks for the negation slash ----
slash_before = df['transcription'].fillna('').apply(lambda t: unicodedata.normalize('NFC', t).count('/')).sum()
slash_after = df['transcription_clean'].str.count('/').sum()
print(f"'/' count before cleaning: {slash_before} | after cleaning: {slash_after}")
assert slash_before == slash_after, "Cleaning removed some '/' characters!"

# '/' should only appear at the START of a word; flag anything else so you can inspect it
odd = df[df['transcription_clean'].str.contains(r'\S/', regex=True)]
print(f"Rows with a '/' in the middle/end of a word: {len(odd)}")
if len(odd):
    print(odd['transcription_clean'].head(5).tolist())

print("Characters removed by cleaning (check this list looks right):", dict(removed_chars))

df['audio_path'] = df.apply(lambda row: os.path.join(row['absolute_dir_path'], row['file_name']), axis=1)
df['file_exists'] = df['audio_path'].apply(os.path.exists)
df = df[df['file_exists'] == True].reset_index(drop=True)
print(f"Total valid audio samples found: {len(df)}")

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Train size: {len(train_df)} | Eval size: {len(eval_df)}")

'/' count before cleaning: 43 | after cleaning: 43
Rows with a '/' in the middle/end of a word: 0
Characters removed by cleaning (check this list looks right): {}
Total valid audio samples found: 418
Train size: 376 | Eval size: 42


In [ ]:
import os
import zipfile
import pandas as pd
import glob

# Path to the new zip file containing Acts 1 to 8
new_zip_path = '/content/drive/MyDrive/corpus/Acts_1_to_8.zip'
new_extract_path = '/content/dataset_acts_1_8'

if not os.path.exists(new_extract_path):
    print("Extracting Acts 1-8 dataset...")
    with zipfile.ZipFile(new_zip_path, 'r') as zip_ref:
        zip_ref.extractall(new_extract_path)
    print("Extraction complete!")

# Search for all metadata.csv files inside Acts_01 to Acts_08
metadata_files = glob.glob(os.path.join(new_extract_path, '**/metadata.csv'), recursive=True)
print(f"Found {len(metadata_files)} metadata files:", metadata_files)

# Combine all CSVs into a single DataFrame with appropriate folder resolution
df_list = []
for file_path in metadata_files:
    temp_df = pd.read_csv(file_path)
    # Retain directory context relative to extraction path
    parent_dir = os.path.dirname(file_path)
    temp_df['absolute_dir_path'] = parent_dir
    df_list.append(temp_df)

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Combined metadata records: {len(combined_df)}")
display(combined_df.head())

Found 8 metadata files: ['/content/dataset_acts_1_8/Acts_1_to_8/Acts_06/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_07/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_01/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_03/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_02/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_04/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_08/metadata.csv', '/content/dataset_acts_1_8/Acts_1_to_8/Acts_05/metadata.csv']
Combined metadata records: 418


,file_name,transcription,absolute_dir_path
0,Acts_06_0001.flac,la̱t cio̱kä bäkɛl kɛn ni̱ni̱ ti̱ti̱ mëë wä ji̱...,/content/dataset_acts_1_8/Acts_1_to_8/Acts_06
1,Acts_06_0002.flac,cu ji̱ gi̱ri̱th göny kɛ kui̱ ji̱ ɣibru ɛn ɣöö ...,/content/dataset_acts_1_8/Acts_1_to_8/Acts_06
2,Acts_06_0003.flac,cu ja̱a̱k da̱ŋ wäl wi̱cdɛ rɛw ji̱ kɔaar yecu d...,/content/dataset_acts_1_8/Acts_1_to_8/Acts_06
3,Acts_06_0004.flac,/ciɛ cuɔ̱ɔ̱ŋ ɛn ɣöö bakɔ la̱t ruac kuɔth päl k...,/content/dataset_acts_1_8/Acts_1_to_8/Acts_06
4,Acts_06_0005.flac,kuany wutni̱ da̱ŋ bärɔw rɛydun ti̱ tekɛ kɛ̈k m...,/content/dataset_acts_1_8/Acts_1_to_8/Acts_06


In [ ]:
# CELL 3b: PROCESS NEW DATASET (ACTS 1 TO 8)
# Clean transcripts, construct absolute audio file paths, verify existences, and split into train/eval datasets.

import os
import re
import unicodedata
from collections import Counter
from sklearn.model_selection import train_test_split

# Characters that are ALWAYS kept, even though they are punctuation/symbols:
#   '/'  = negation marker at the start of a word
PROTECTED = {'/'}

removed_chars = Counter()

def clean_and_normalize(text):
    if not isinstance(text, str):
        return ""
    # NFC keeps precomposed diacritics consistent
    text = unicodedata.normalize('NFC', text)
    kept = []
    for ch in text:
        cat = unicodedata.category(ch)
        if (
            ch in PROTECTED          # the negation slash
            or ch.isalpha()          # letters incl. ɛ ɔ ŋ ɣ ä ë ö ...
            or cat.startswith('M')   # combining marks
            or ch.isspace()
        ):
            kept.append(ch)
        else:
            removed_chars[ch] += 1   # punctuation, digits, symbols get dropped
    text = ''.join(kept)
    return re.sub(r'\s+', ' ', text).strip()

# We will use the same clean_and_normalize function defined earlier
combined_df['transcription_clean'] = combined_df['transcription'].apply(clean_and_normalize)

# Construct absolute paths to the .flac audio files
combined_df['audio_path'] = combined_df.apply(
    lambda row: os.path.join(row['absolute_dir_path'], row['file_name']),
    axis=1
)

# Verify that all files exist locally
combined_df['file_exists'] = combined_df['audio_path'].apply(os.path.exists)
invalid_files = combined_df[combined_df['file_exists'] == False]
print(f"Missing files count: {len(invalid_files)}")

# Filter to include only valid files
df_filtered = combined_df[combined_df['file_exists'] == True].reset_index(drop=True)
print(f"Total valid audio samples found: {len(df_filtered)}")

# Overwrite train_df and eval_df with the new larger dataset
train_df, eval_df = train_test_split(df_filtered, test_size=0.1, random_state=42)
print(f"New Train size: {len(train_df)} | New Eval size: {len(eval_df)}")

Missing files count: 0
Total valid audio samples found: 418
New Train size: 376 | New Eval size: 42


In [ ]:
# CELL 4: CUSTOM NUER VOCABULARY AND TOKENIZER CREATION
import json
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# Build vocab from ALL transcripts in the new filtered dataset
all_text = " ".join(df_filtered['transcription_clean'].tolist())
vocab_list = sorted(set(all_text))
vocab_dict = {v: i for i, v in enumerate(vocab_list)}

if ' ' in vocab_dict:
    vocab_dict['|'] = vocab_dict.pop(' ')

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open('vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print("Nuer Vocabulary Size:", len(vocab_dict))
print("Vocabulary Map:", vocab_dict)

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# ---- Tokenizer round-trip check: '/' must survive encode -> decode ----
assert '/' in vocab_dict, "'/' is missing from the vocabulary!"
sample = next((t for t in df_filtered['transcription_clean'] if '/' in t), df_filtered['transcription_clean'][0])
decoded = processor.tokenizer.decode(processor.tokenizer(text=sample).input_ids, group_tokens=False)
print("Original:", sample)
print("Decoded :", decoded)
assert decoded == sample, "Round trip changed the text (check for [UNK] or dropped characters)"
print("OK: negation slash survives tokenization")

Nuer Vocabulary Size: 33
Vocabulary Map: {'/': 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5, 'e': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'r': 17, 't': 18, 'u': 19, 'w': 20, 'y': 21, 'ä': 22, 'ë': 23, 'ö': 24, 'ŋ': 25, 'ɔ': 26, 'ɛ': 27, 'ɣ': 28, '̈': 29, '̱': 30, '|': 0, '[UNK]': 31, '[PAD]': 32}
Original: cu ji̱ gi̱ri̱th göny kɛ kui̱ ji̱ ɣibru ɛn ɣöö /ca män jɔɔkni̱kiɛn tit kɛ dääk mi̱thni̱ ni̱ ciaŋ
Decoded : cu ji̱ gi̱ri̱th göny kɛ kui̱ ji̱ ɣibru ɛn ɣöö /ca män jɔɔkni̱kiɛn tit kɛ dääk mi̱thni̱ ni̱ ciaŋ
OK: negation slash survives tokenization


In [ ]:
# CELL 5: HUGGING FACE DATASET CONVERSION & PREPROCESSING
from datasets import Dataset, Audio

# Convert pandas dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[['audio_path', 'transcription_clean']])
eval_dataset = Dataset.from_pandas(eval_df[['audio_path', 'transcription_clean']])

# Cast file paths to Hugging Face Audio Feature for automatic loading and resample to 16kHz
train_dataset = train_dataset.cast_column("audio_path", Audio(sampling_rate=16000))
eval_dataset = eval_dataset.cast_column("audio_path", Audio(sampling_rate=16000))

# Preprocessing mapping function
def prepare_dataset(batch):
    audio = batch["audio_path"]

    # Process raw audio array
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    # Process target labels (character token IDs) without as_target_processor()
    batch["labels"] = processor(text=batch["transcription_clean"]).input_ids
    return batch

# Map datasets
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=eval_dataset.column_names)

print("Sample processed inputs:", train_dataset[0].keys())

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Sample processed inputs: dict_keys(['input_values', 'input_length', 'labels'])


In [ ]:
# CELL 6: CTC DATA COLLATOR CLASS
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Separate inputs and labels since they require different padding strategies
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Pad audio inputs
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Pad label IDs using the tokenizer explicitly
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Replace padding labels with -100 to ignore loss calculations
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# CELL 7: MMS-1B-ALL MODEL + FRESH NUER ADAPTER
import torch, gc
from transformers import Wav2Vec2ForCTC

gc.collect()
torch.cuda.empty_cache()
device = "cuda"

# mms-1b-all has a backbone already fine-tuned for CTC, which is what adapters are designed to sit on.
# ignore_mismatched_sizes re-initializes lm_head for our own Nuer vocab.
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-1b-all",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)

model.init_adapter_layers()   # fresh adapter layers for the new vocab
model.freeze_base_model()     # freeze backbone

# Train only adapter layers + lm_head
for p in model._get_adapters().values():
    p.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([35, 1280])
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([35])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable params: 2,196,003 / 964,693,411


In [ ]:
# CELL 8: METRICS, TRAINING ARGUMENTS & EXECUTION
import evaluate
import numpy as np
import torch, gc
from transformers import TrainingArguments, Trainer

gc.collect()
torch.cuda.empty_cache()

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
    }

training_args = TrainingArguments(
    output_dir="./mms-1b-nuer-adapter",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,     # effective batch 8 -> ~12 optimizer steps/epoch
    eval_strategy="epoch",
    save_strategy="no",                # adapter is saved manually in the next cell (avoids 3.8GB checkpoints)
    learning_rate=1e-3,                # adapter-style LR; 1e-4 was too low for ~60 total steps
    weight_decay=0.005,
    num_train_epochs=30,
    fp16=True,
    gradient_checkpointing=False,      # if you hit CUDA OOM: set True and uncomment the next line
    # gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 34, 'bos_token_id': 33}.


Epoch,Training Loss,Validation Loss,Wer,Cer
1,8.489352,4.678275,0.997028,0.889893
2,7.280499,4.279014,1.000000,0.928318
3,1.090989,0.364124,0.384844,0.105586
4,0.772803,0.287026,0.309064,0.087504
5,0.684135,0.266270,0.337296,0.087181
6,0.633309,0.232149,0.276374,0.073297
7,0.588770,0.219959,0.274889,0.070714
8,0.511933,0.215323,0.263001,0.067162
9,0.539281,0.228662,0.294205,0.073297
10,0.416070,0.219679,0.289747,0.071359


TrainOutput(global_step=1410, training_loss=1.0630612025869655, metrics={'train_runtime': 3158.0636, 'train_samples_per_second': 3.572, 'train_steps_per_second': 0.446, 'total_flos': 9.138871919532603e+18, 'train_loss': 1.0630612025869655, 'epoch': 30.0})

In [ ]:
# CELL 9: SAVE ONLY THE ADAPTER (a few MB) + PROCESSOR TO DRIVE
from safetensors.torch import save_file

save_dir = "/content/drive/MyDrive/nuer_mms_adapter"
os.makedirs(save_dir, exist_ok=True)

weights = {k: v.detach().cpu().contiguous() for k, v in model._get_adapters().items()}
save_file(weights, os.path.join(save_dir, "adapter.nus.safetensors"))
processor.save_pretrained(save_dir)
print("Saved to", save_dir)

Saved to /content/drive/MyDrive/nuer_mms_adapter


In [ ]:
# CELL 10: QUICK SANITY CHECK ON EVAL CLIPS
model.eval()
for i in range(min(5, len(eval_dataset))):
    x = torch.tensor(eval_dataset[i]["input_values"]).unsqueeze(0).to(device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).logits
    pred = processor.batch_decode(torch.argmax(logits, dim=-1))[0]
    ref = processor.decode(eval_dataset[i]["labels"], group_tokens=False)
    print(f"REF : {ref}\nPRED: {pred}\n")

REF : kɛ ɣöö ŋotdɛ /ken yiëë ni̱ ben kä kɛ kä ca lak ni̱kɛ kɛ ciöt yecu keri̱tho kärɔa
PRED: kɛ ɣöö ŋottdɛ ken yiëëni̱ ben kä kɛ kä ca lakni̱kɛ kɛ ciöt yecu keri̱tho kɛ rɔa

REF : ka̱mɛ ɣä ɛn buɔ̱m ɛmɛ bɔ̱ kɛ ɣöö ram mɔ wa̱a̱ tetdä kuëŋ wi̱i̱dɛ mɔ derɛ yiëë in gɔaa in rɛl rɔ jek
PRED: kämɛ ɣä ɛn buɔ̱m ɛmɛ bɔ̱ kɛ ɣöö ramɔ wa̱at tetdä kuëŋ wi̱i̱t ɛmɔ därä yiëë in gɔaa in rɛl rɔ jäk

REF : gääkä kä te yiëën cuŋä thukni̱ thiekni̱ kä mëë ca̱kɔ kɛ lɛp thilɛ ram mi̱ ca̱kɔ jek thi̱n
PRED: gääkä kä te yiëën cuŋä thukni̱ thiäkni̱ kä mëë /cakɔ kɛ lɛp pɛlɛ ram mi ca̱kɔ jek thi̱n

REF : kɛ tetdɛ kä duundɛ ɣöö /ken kɛn ɛ ŋa̱c mëë ruɔ̱n cuɛ ji̱ i̱thruɛl
PRED: kɛ tetdɛ kä duundɛ ɣöö /kɛn kɛnɛ ŋa̱c mëë ruɔ̱n cuɛ ji̱ i̱thruɛl

REF : kä cuɛ jɛ jiac nhial kä cu ciökɛ kɛnɛ copkɛ buɔ̱m ni̱ kä kɛɛl kä cuɛ rɔ yuɔr nhial
PRED: kä cuɛ jɛ jiac nhial kä cu ciökɛ kɛnɛ copkɛ buɔm ni̱kä kɛɛl kä cuɛ rɛ yuɛt nhial

